# 05 · Unit economics marítimos

Normaliza ingresos y costes por salida, pasajero y asiento-milla náutica para comparar rutas de distinta escala.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## KPIs unitarios por ruta

In [2]:
a=pd.read_csv(RAW/'fact_finance_actual.csv',parse_dates=['month'])
u=a.groupby('route_id',as_index=False).agg(revenue=('revenue','sum'),total_cost=('variable_cost','sum'),fixed=('fixed_allocated','sum'),voyages=('voyages','sum'),passengers=('passengers','sum'),available_seats=('available_seats','sum'),seat_nm=('distance_nm',lambda s:0))
# El asiento-milla se calcula registro a registro para respetar capacidad y distancia.
seat_nm=a.assign(seat_nm=a.available_seats*a.distance_nm).groupby('route_id').seat_nm.sum()
u['total_cost']=u.total_cost+u.fixed
u['revenue_per_sailing']=u.revenue/u.voyages
u['cost_per_sailing']=u.total_cost/u.voyages
u['revenue_per_passenger']=u.revenue/u.passengers
u['rask']=u.revenue/u.route_id.map(seat_nm)
u['cask']=u.total_cost/u.route_id.map(seat_nm)
u['spread_rask_cask']=u.rask-u.cask
u.to_csv(TABLES/'05_unit_economics_route.csv',index=False)
display(u.sort_values('spread_rask_cask',ascending=False))

  route_id       revenue    total_cost  ...  rask  cask  spread_rask_cask
4  DEN-PMI 82,032,940.36 67,600,601.54  ...  0.38  0.31              0.07
2  DEN-FOR 71,058,312.21 65,186,492.67  ...  0.61  0.56              0.05
3  DEN-IBZ 73,159,514.13 69,119,963.23  ...  0.57  0.54              0.03
5  VAL-IBZ 75,664,657.28 71,455,218.57  ...  0.42  0.39              0.02
1  BCN-PMI 86,062,252.40 82,116,770.83  ...  0.35  0.34              0.02
6  VAL-PMI 90,299,936.26 86,160,655.72  ...  0.30  0.28              0.01
0  BCN-IBZ 95,828,871.62 92,882,094.04  ...  0.29  0.28              0.01

[7 rows x 14 columns]


## RASK frente a CASK

In [3]:
plot=u.sort_values('spread_rask_cask')
y=np.arange(len(plot)); h=.36
plt.figure(figsize=(10,6)); plt.barh(y-h/2,plot.rask,h,label='RASK',color='#1f6feb'); plt.barh(y+h/2,plot.cask,h,label='CASK',color='#f59e0b'); plt.yticks(y,plot.route_id); plt.xlabel('€/asiento-milla náutica'); plt.title('Ingresos y coste unitario por ruta'); plt.legend()
plt.tight_layout(); plt.savefig(FIGURES/'05_rask_cask.png',dpi=180,bbox_inches='tight'); plt.show()

## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.